<a href="https://colab.research.google.com/github/justynigam/DS-Task/blob/main/DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
trader_df = pd.read_csv('/content/DStask/historical_data.csv')
sentiment_df = pd.read_csv('/content/DStask/fear_greed_index.csv')

In [ ]:
trader_df.columns = trader_df.columns.str.lower().str.replace(' ', '_')

In [ ]:
trader_df.head()

In [ ]:
trader_df['timestamp_ist'] = pd.to_datetime(trader_df['timestamp_ist'], format='%d-%m-%Y %H:%M')
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

In [ ]:
trader_df['date'] = trader_df['timestamp_ist'].dt.date
trader_df['date'] = pd.to_datetime(trader_df['date'])

In [ ]:
merged_df = pd.merge(trader_df, sentiment_df, on='date', how='inner')

In [ ]:
if 'leverage' not in merged_df.columns:
    print("Warning: 'leverage' column not found in historical_data.csv. Skipping leverage analysis.")
    leverage_analysis_possible = False
else:
    leverage_analysis_possible = True

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='classification', data=merged_df, order=merged_df['classification'].value_counts().index)
plt.title('Distribution of Market Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Number of Trades')
plt.savefig('sentiment_distribution.png')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(merged_df['closed_pnl'], bins=100, kde=True)
plt.title('Distribution of Trader Closed PnL')
plt.xlabel('Closed PnL')
plt.ylabel('Frequency')
plt.xlim(-500, 500)
plt.tight_layout()
plt.savefig('pnl_distribution.png')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='classification', y='closed_pnl', data=merged_df, order=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'])
plt.title('Trader PnL by Market Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Closed PnL')
plt.ylim(-500, 500)
plt.tight_layout()
plt.savefig('pnl_by_sentiment.png')
plt.show()

In [ ]:
if leverage_analysis_possible:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='classification', y='leverage', data=merged_df, order=['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed'])
    plt.title('Leverage Used by Market Sentiment')
    plt.xlabel('Sentiment')
    plt.ylabel('Leverage')
    plt.tight_layout()
    plt.savefig('leverage_by_sentiment.png')
    plt.show()

In [ ]:
side_sentiment = merged_df.groupby(['classification', 'side']).size().unstack(fill_value=0)
side_sentiment.plot(kind='bar', stacked=True, figsize=(12, 7))
plt.title('Trading Side by Market Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Number of Trades')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('side_by_sentiment.png')
plt.show()

In [ ]:
columns_to_check = ['closed_pnl', 'classification', 'side']
if leverage_analysis_possible:
    columns_to_check.append('leverage')


columns_to_check = [col for col in columns_to_check if col in merged_df.columns]

merged_df.dropna(subset=columns_to_check, inplace=True)

In [ ]:
merged_df['is_profitable'] = (merged_df['closed_pnl'] > 0).astype(int)

In [ ]:
features = ['classification', 'side', 'size_usd']
X = merged_df[features]
y = merged_df['is_profitable']

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


categorical_features = ['classification', 'side']
encoder = OneHotEncoder(handle_unknown='ignore')
X_encoded = pd.DataFrame(encoder.fit_transform(X[categorical_features]).toarray(),
                         columns=encoder.get_feature_names_out(categorical_features))

X = pd.concat([X.drop(columns=categorical_features).reset_index(drop=True), X_encoded], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)


y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("--- Predictive Model Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Profitable', 'Profitable']))

In [ ]:
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Profitable', 'Profitable'], yticklabels=['Not Profitable', 'Profitable'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')
plt.show()